# Analisis Data Fasilitas Kesehatan Puskesmas — DKI Jakarta

**Tujuan:** menganalisis distribusi geografis dan kualitas data fasilitas Puskesmas dari dataset Satu Data Jakarta menggunakan Python di Google Colab.

**Pertanyaan analitis**
1. Bagaimana jumlah catatan Puskesmas tersebar antarwilayah administrasi?
2. Apakah terdapat duplikasi, nilai kosong, atau masalah format pada kolom penting?
3. Bagaimana sebaran lokasi berdasarkan longitude dan latitude?
4. Data tambahan apa yang diperlukan untuk menilai pemerataan akses?

**Catatan penting:** file yang diunggah berekstensi `.xls`, tetapi kontennya berupa tabel HTML. Karena itu, notebook membaca tabel menggunakan `pandas.read_html()`, bukan `pd.read_excel()`. Analisis ini bersifat deskriptif dan tidak menyimpulkan pemerataan akses hanya dari jumlah fasilitas.

## 1. Upload file dan import library

In [ ]:
from google.colab import files
uploaded = files.upload()  # Pilih file dataset Puskesmas yang diunduh dari portal

import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

filename = next(iter(uploaded))
print("File:", filename)
print("Ukuran file (bytes):", len(uploaded[filename]))

## 2. Baca dan inspeksi data

In [ ]:
# File sumber berekstensi .xls, tetapi isi aktualnya adalah tabel HTML
tables = pd.read_html(io.BytesIO(uploaded[filename]))
print("Jumlah tabel ditemukan:", len(tables))

df_raw = tables[0].copy()
print("Ukuran data:", df_raw.shape)
display(df_raw.head())
display(df_raw.dtypes)

In [ ]:
# Nama kolom, tipe data, dan ringkasan awal
print("Kolom:")
print(df_raw.columns.tolist())

print("\\nInfo dataset:")
df_raw.info()

print("\\nRingkasan nilai unik per kolom:")
display(df_raw.nunique(dropna=False).sort_values(ascending=False).to_frame("jumlah_nilai_unik"))

## 3. Data quality assessment

In [ ]:
# Periksa nilai kosong dan persentasenya
quality = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
    "unique_values": df_raw.nunique(dropna=False)
}).sort_values("missing_pct", ascending=False)

display(quality)

print("Duplikasi seluruh baris:", df_raw.duplicated().sum())
print("Duplikasi ID:", df_raw["id_puskesmas"].duplicated().sum())
print("Duplikasi nama:", df_raw["nama_puskesmas"].astype("string").str.strip().str.upper().duplicated().sum())

## 4. Cleaning ringan dan validasi

In [ ]:
df = df_raw.copy()

# Rapikan spasi dan konsistensi teks; jangan mengubah data mentah
text_cols = [
    "nama_puskesmas", "wilayah_puskesmas", "alamat_puskesmas",
    "no_telepon", "email", "website_puskesmas"
]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace({"": pd.NA, "-": pd.NA, "N/A": pd.NA, "NA": pd.NA})

# Pastikan kolom numerik bertipe angka
for col in ["periode_data", "id_puskesmas", "longitude_puskesmas", "latitude_puskesmas"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Kolom pemeriksaan koordinat
df["koordinat_valid"] = (
    df["longitude_puskesmas"].between(106.0, 107.5)
    & df["latitude_puskesmas"].between(-7.0, -5.0)
)

# Pemeriksaan format dasar email: hanya indikasi format, bukan verifikasi email aktif
email = df["email"].astype("string").str.strip()
df["email_format_ok"] = email.str.match(r"^[^\\s@]+@[^\\s@]+\\.[^\\s@]+$", na=False)

# Pemeriksaan URL website: domain tanpa https:// tetap bisa merupakan alamat yang valid
website = df["website_puskesmas"].astype("string").str.strip()
df["website_has_scheme"] = website.str.match(r"^https?://", case=False, na=False)

display(df.head())
print("Periode yang tersedia:", sorted(df["periode_data"].dropna().unique().tolist()))
print("Koordinat di dalam rentang pemeriksaan Jakarta:", int(df["koordinat_valid"].sum()), "dari", len(df))

## 5. Standardisasi wilayah dan distribusi fasilitas

In [ ]:
# Buat label wilayah yang lebih ringkas untuk grafik
df["wilayah_ringkas"] = (
    df["wilayah_puskesmas"]
      .astype("string")
      .str.replace("KOTA ADM. ", "", regex=False)
      .str.replace("KAB. ADM. ", "", regex=False)
      .str.strip()
      .str.title()
)

distribusi = (
    df.groupby("wilayah_ringkas", dropna=False)
      .agg(jumlah_puskesmas=("id_puskesmas", "nunique"))
      .sort_values("jumlah_puskesmas", ascending=False)
      .reset_index()
)
distribusi["persentase"] = (
    distribusi["jumlah_puskesmas"] / distribusi["jumlah_puskesmas"].sum() * 100
).round(1)

display(distribusi)
print("Total ID Puskesmas unik:", df["id_puskesmas"].nunique())

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=distribusi, x="jumlah_puskesmas", y="wilayah_ringkas")
ax.set_title("Jumlah Catatan Puskesmas per Wilayah Administrasi")
ax.set_xlabel("Jumlah Puskesmas (ID unik)")
ax.set_ylabel("Wilayah")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)
plt.tight_layout()
plt.show()

## 6. Pemeriksaan kualitas data per wilayah

In [ ]:
quality_by_region = (
    df.groupby("wilayah_ringkas")
      .agg(
          total=("id_puskesmas", "size"),
          koordinat_valid=("koordinat_valid", "sum"),
          email_format_ok=("email_format_ok", "sum"),
          website_dengan_scheme=("website_has_scheme", "sum")
      )
      .reset_index()
)
quality_by_region["email_format_issue"] = quality_by_region["total"] - quality_by_region["email_format_ok"]
quality_by_region["koordinat_issue"] = quality_by_region["total"] - quality_by_region["koordinat_valid"]
display(quality_by_region.sort_values("total", ascending=False))

## 7. Analisis kontak: indikasi format, bukan validitas layanan

In [ ]:
contact_summary = pd.DataFrame({
    "indikator": [
        "Telepon terisi",
        "Email terisi",
        "Email sesuai pola dasar",
        "Website terisi",
        "Website memakai http/https"
    ],
    "jumlah": [
        df["no_telepon"].notna().sum(),
        df["email"].notna().sum(),
        df["email_format_ok"].sum(),
        df["website_puskesmas"].notna().sum(),
        df["website_has_scheme"].sum()
    ]
})
contact_summary["persentase"] = (contact_summary["jumlah"] / len(df) * 100).round(1)
display(contact_summary)

# Tinjau baris yang perlu pemeriksaan manual
display(df.loc[~df["email_format_ok"], [
    "nama_puskesmas", "wilayah_puskesmas", "email"
]])

## 8. Visualisasi lokasi berdasarkan koordinat

In [ ]:
# Scatter plot longitude-latitude; bukan peta jalan atau batas administrasi
plt.figure(figsize=(8, 8))
ax = sns.scatterplot(
    data=df,
    x="longitude_puskesmas",
    y="latitude_puskesmas",
    hue="wilayah_ringkas",
    s=75
)
ax.set_title("Sebaran Koordinat Puskesmas DKI Jakarta")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Wilayah", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 9. Susun temuan otomatis awal

In [ ]:
top_region = distribusi.iloc[0]
bottom_region = distribusi.iloc[-1]
email_issues = int((~df["email_format_ok"]).sum())
coord_issues = int((~df["koordinat_valid"]).sum())

print(f"Jumlah baris: {len(df)}")
print(f"Jumlah ID unik: {df['id_puskesmas'].nunique()}")
print(f"Wilayah dengan catatan terbanyak: {top_region['wilayah_ringkas']} ({top_region['jumlah_puskesmas']})")
print(f"Wilayah dengan catatan paling sedikit: {bottom_region['wilayah_ringkas']} ({bottom_region['jumlah_puskesmas']})")
print(f"Email yang gagal pemeriksaan format dasar: {email_issues}")
print(f"Koordinat di luar rentang pemeriksaan: {coord_issues}")

print("\\nInterpretasi:")
print("- Perbedaan jumlah fasilitas bersifat deskriptif; belum menunjukkan akses yang lebih baik atau buruk.")
print("- Format email yang gagal harus diperiksa manual dan tidak otomatis berarti email tidak aktif.")
print("- Untuk menilai pemerataan, tambahkan jumlah penduduk per wilayah dan samakan tahun serta tingkat geografisnya.")

## 10. Ekspor data bersih dan tabel ringkasan

In [ ]:
df.to_csv("puskesmas_cleaned.csv", index=False)
distribusi.to_csv("puskesmas_distribution_by_region.csv", index=False)
quality_by_region.to_csv("puskesmas_quality_by_region.csv", index=False)

from google.colab import files
files.download("puskesmas_cleaned.csv")
files.download("puskesmas_distribution_by_region.csv")
files.download("puskesmas_quality_by_region.csv")

## 11. Keterbatasan dan analisis lanjutan

- Dataset ini merupakan snapshot periode 2025; tidak dapat digunakan untuk tren waktu.
- Jumlah catatan fasilitas bukan ukuran aksesibilitas, kapasitas, mutu, atau utilisasi layanan.
- Scatter plot koordinat menunjukkan posisi relatif, tetapi bukan jarak perjalanan atau jangkauan layanan.
- Untuk analisis penduduk per Puskesmas, gabungkan data penduduk dengan tingkat wilayah dan tahun yang cocok.
- Sebelum mempublikasikan, verifikasi definisi unit fasilitas dan metadata dataset di portal sumber.
- Dokumentasikan sumber, tanggal akses, aturan cleaning, dan semua asumsi di README GitHub.